In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler
from xgboost import XGBRegressor 
from sklearn.metrics import mean_absolute_error, r2_score

# ============================================================
# 🖥️ 1. PREPARAÇÃO, ENGENHARIA E SPLIT DOS DADOS
# ============================================================
df = pd.read_parquet('../../dataset/data.parquet', engine='fastparquet')
df = df[df['Desired_Savings'] > 0].copy()

# Cálculo das métricas de comportamento
df['perc_nao_essenciais'] = (df['Eating_Out'] + df['Entertainment']) / df['Income']
df['perc_emprestimo'] = df['Loan_Repayment'] / df['Income']

# TARGET CONTÍNUO: Mapeamento de Risco de 0 a 100
df['Risk_Score'] = (
    (df['perc_emprestimo'] * 0.4) + 
    (df['perc_nao_essenciais'] * 0.3) + 
    ((df['Potential_Savings_Groceries'] / df['Income']) * 0.3)
) * 100  



X = df[features_percentuais]
y = df['Risk_Score']

# Split de Regressão (Sem o parâmetro stratify)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Escalonamento RobustScaler
scaler = RobustScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns)
X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=X_test.columns)

# ============================================================
# ⚙️ 2. TREINAMENTO DO REGRESSOR E PREVISÃO
# ============================================================
modelo_risco = XGBRegressor(
    random_state=42, max_depth=6, learning_rate=0.1, n_estimators=100, eval_metric='rmse'
)
modelo_risco.fit(X_train_scaled, y_train)

# Previsão das notas contínuas para a base de teste
y_pred_score = modelo_risco.predict(X_test_scaled)

# ============================================================
# 📊 3. CÓDIGO PARA VER A QUANTIDADE DE USUÁRIOS DE CADA RISCO
# ============================================================
print("="*65)
print("📊 VOLUMETRIA E SEGMENTAÇÃO COMERCIAL DE CRÉDITO (BASE DE TESTE)")
print("="*65)

# Estruturamos um DataFrame de análise contendo os valores reais e previstos
df_analise = pd.DataFrame({
    'Score_Real': y_test.values,
    'Score_Previsto': y_pred_score
})

# Função de Regra de Negócio para mapear as faixas de risco
def segmentar_risco(score):
    if score <= 15:
        return 'Baixo Risco (Verde)'
    elif score <= 35:
        return 'Médio Risco (Amarelo)'
    else:
        return 'Alto Risco (Vermelho)'

# Aplicamos a segmentação nas notas reais e nas notas previstas pelo modelo
df_analise['Segmento_Real'] = df_analise['Score_Real'].apply(segmentar_risco)
df_analise['Segmento_Previsto'] = df_analise['Score_Previsto'].apply(segmentar_risco)

# Exibe a contagem absoluta e percentual do que o modelo previu
contagem_absoluta = df_analise['Segmento_Previsto'].value_counts()
contagem_percentual = df_analise['Segmento_Previsto'].value_counts(normalize=True) * 100

for segmento in contagem_absoluta.index:
    print(f"🔹 {segmento}: {contagem_absoluta[segmento]} usuários ({contagem_percentual[segmento]:.2f}%)")

# ============================================================
# 🎛️ CORREÇÃO DA REGRA DE NEGÓCIO: CORTES DINÂMICOS POR PERCENTIL
# ============================================================
print("\n" + "="*65)
print("🎯 NOVA SIMULAÇÃO DE SEGMENTAÇÃO CORRIGIDA (CORTES DINÂMICOS)")
print("="*65)

df_producao = pd.DataFrame({
    'Score_Real': y_test.values,
    'Score_Previsto': y_pred_score
})

# MATEMÁTICA SÊNIOR: Descobrimos onde estão os limites das fatias da população prevista
# Corte 1 (50% mais seguros): Pega o valor que separa a metade inferior dos dados
corte_medio_risco = np.percentile(y_pred_score, 50)

# Corte 2 (Os 15% mais perigosos): Pega o início do topo da pirâmide de risco
corte_alto_risco = np.percentile(y_pred_score, 85)

print(f"⚙️ Novas réguas calculadas para a sua população:")
print(f"   🟢 Até {corte_medio_risco:.2f} pontos -> Baixo Risco")
print(f"   🟡 De {corte_medio_risco:.2f} até {corte_alto_risco:.2f} pontos -> Médio Risco")
print(f"   🔴 Acima de {corte_alto_risco:.2f} pontos -> Alto Risco\n")

# Nova função de mapeamento baseada nas réguas populacionais reais
def segmentar_risco_dinamico(score):
    if score <= corte_medio_risco:
        return 'Baixo Risco (Verde)'
    elif score <= corte_alto_risco:
        return 'Médio Risco (Amarelo)'
    else:
        return 'Alto Risco (Vermelho)'

df_producao['Segmento_Dinamico'] = df_producao['Score_Previsto'].apply(segmentar_risco_dinamico)

# Exibe a volumetria corrigida e distribuída
contagem = df_producao['Segmento_Dinamico'].value_counts()
percentual = df_producao['Segmento_Dinamico'].value_counts(normalize=True) * 100

for seg in contagem.index:
    print(f"🔹 {seg}: {contagem[seg]} usuários ({percentual[seg]:.2f}%)")

KeyError: "['Rent_Ratio', 'Healthcare_Ratio', 'Education_Ratio', 'Groceries_Ratio', 'Transport_Ratio', 'Utilities_Ratio', 'Insurance_Ratio'] not in index"

In [4]:
import numpy as np
import pandas as pd
from sklearn.metrics import classification_report

# 1. Criamos o gabarito real binário usando a mesma lógica de corte (Percentil 50 da base real)
corte_real_50 = np.percentile(y_test, 50)
y_test_binario = np.where(y_test >= corte_real_50, 1, 0)

# 2. Criamos a previsão binária baseada no corte que o seu XGBoost gerou (4.72)
# Tudo que for maior que 4.72 vira risco/vulnerável (1), o que for menor vira seguro (0)
y_pred_binario = np.where(y_pred_score > 4.72, 1, 0)

print("="*65)
print("📊 MÉTRICAS DE CLASSIFICAÇÃO EQUIVALENTES DO XGBOOST REGRESSOR")
print("="*65)
print(classification_report(y_test_binario, y_pred_binario, target_names=['Seguro', 'Vulnerável']))

📊 MÉTRICAS DE CLASSIFICAÇÃO EQUIVALENTES DO XGBOOST REGRESSOR
              precision    recall  f1-score   support

      Seguro       0.51      0.51      0.51      1989
  Vulnerável       0.51      0.51      0.51      1989

    accuracy                           0.51      3978
   macro avg       0.51      0.51      0.51      3978
weighted avg       0.51      0.51      0.51      3978



In [6]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler
from xgboost import XGBRegressor 
from sklearn.metrics import mean_absolute_error, r2_score, classification_report

# ============================================================
# 🖥️ 1. CARREGAMENTO E ENGENHARIA DE FEATURES (SEM LEAKAGE)
# ============================================================
df = pd.read_parquet('../../dataset/data.parquet', engine='fastparquet')
df = df[df['Desired_Savings'] > 0].copy()

# Cálculo dos componentes do Target
df['perc_nao_essenciais'] = (df['Eating_Out'] + df['Entertainment']) / df['Income']
df['perc_emprestimo'] = df['Loan_Repayment'] / df['Income']

# TARGET CONTÍNUO: Mapeamento de Risco de 0 a 100
df['Risk_Score'] = (
    (df['perc_emprestimo'] * 0.4) + 
    (df['perc_nao_essenciais'] * 0.3) + 
    ((df['Potential_Savings_Groceries'] / df['Income']) * 0.3)
) * 100  

# --- ENGENHARIA DE FEATURES PERMITIDA (Dando olhos ao modelo de forma limpa) ---
# 1. Renda Per Capita (Mapeia a densidade e o peso familiar sobre o bolso)
df['Renda_Per_Capita'] = df['Income'] / (df['Dependents'] + 1)

# 2. Saldo Básico Livre (O que sobra do salário tirando o consumo básico de subsistência)
df['Saldo_Básico_Livre'] = df['Income'] - (df['Rent'] + df['Healthcare'] + df['Education'] + df['Utilities'])

# Calculando as proporções das despesas fixas tradicionais
df['Rent_Ratio'] = df['Rent'] / df['Income']
df['Healthcare_Ratio'] = df['Healthcare'] / df['Income']
df['Education_Ratio'] = df['Education'] / df['Income']
df['Groceries_Ratio'] = df['Groceries'] / df['Income']
df['Transport_Ratio'] = df['Transport'] / df['Income']
df['Utilities_Ratio'] = df['Utilities'] / df['Income']
df['Insurance_Ratio'] = df['Insurance'] / df['Income']

# SELEÇÃO DE FEATURES FINAL (Protegida contra Target Leakage)
features_percentuais = [
    'Age', 
    'Dependents', 
    'Income',               # Injetamos o volume total da renda
    'Renda_Per_Capita',     # Injetamos o contexto de tamanho da família
    'Saldo_Básico_Livre',   # Injetamos o fôlego financeiro estimado
    'Rent_Ratio', 
    'Healthcare_Ratio', 
    'Education_Ratio', 
    'Groceries_Ratio',
    'Transport_Ratio',
    'Utilities_Ratio',
    'Insurance_Ratio'
    # ATENÇÃO: 'perc_emprestimo' e 'perc_nao_essenciais' FICAM FORA para evitar Vazamento!
]

X = df[features_percentuais]
y = df['Risk_Score']

# Split de Regressão simples (Sem o parâmetro stratify)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Escalonamento Robusto para blindar a matriz contra distorções de outliers
scaler = RobustScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns)
X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=X_test.columns)


# ============================================================
# ⚙️ 2. TREINAMENTO DO MODELO DE REGRESSÃO
# ============================================================
print("🚀 Treinando o XGBRegressor com novas Features de contexto...\n")

modelo_risco = XGBRegressor(
    random_state=42, 
    max_depth=6, 
    learning_rate=0.1, 
    n_estimators=100, 
    eval_metric='rmse'
)
modelo_risco.fit(X_train_scaled, y_train)

# Previsão das notas contínuas para a base de teste
y_pred_score = modelo_risco.predict(X_test_scaled)


# ============================================================
# 📊 3. NOVA VOLUMETRIA POR CORTES DINÂMICOS (PERCENTILE)
# ============================================================
print("="*65)
print("🎯 NOVA SIMULAÇÃO DE SEGMENTAÇÃO CORRIGIDA (CORTES DINÂMICOS)")
print("="*65)

df_producao = pd.DataFrame({
    'Score_Real': y_test.values,
    'Score_Previsto': y_pred_score
})

# Calculamos as réguas com base na nova distribuição prevista
corte_medio_risco = np.percentile(y_pred_score, 50)
corte_alto_risco = np.percentile(y_pred_score, 85)

print(f"⚙️ Novas réguas calculadas para a sua população:")
print(f"   🟢 Até {corte_medio_risco:.2f} pontos -> Baixo Risco")
print(f"   🟡 De {corte_medio_risco:.2f} até {corte_alto_risco:.2f} pontos -> Médio Risco")
print(f"   🔴 Acima de {corte_alto_risco:.2f} pontos -> Alto Risco\n")

def segmentar_risco_dinamico(score):
    if score <= corte_medio_risco:
        return 'Baixo Risco (Verde)'
    elif score <= corte_alto_risco:
        return 'Médio Risco (Amarelo)'
    else:
        return 'Alto Risco (Vermelho)'

df_producao['Segmento_Dinamico'] = df_producao['Score_Previsto'].apply(segmentar_risco_dinamico)

# Exibe a volumetria corrigida e distribuída
contagem = df_producao['Segmento_Dinamico'].value_counts()
percentual = df_producao['Segmento_Dinamico'].value_counts(normalize=True) * 100

for seg in contagem.index:
    print(f"🔹 {seg}: {contagem[seg]} usuários ({percentual[seg]:.2f}%)")


# ============================================================
# ⚖️ 4. TRADUÇÃO PARA MÉTRICAS DE CLASSIFICAÇÃO EQUIVALENTES
# ============================================================
# Criamos o gabarito real binário usando a mesma lógica de corte (Percentil 50 da base real)
corte_real_50 = np.percentile(y_test, 50)
y_test_binario = np.where(y_test >= corte_real_50, 1, 0)

# Criamos a previsão binária baseada na régua dinâmica que o seu XGBoost gerou
y_pred_binario = np.where(y_pred_score > corte_medio_risco, 1, 0)

print("\n" + "="*65)
print("📊 MÉTRICAS DE CLASSIFICAÇÃO EQUIVALENTES (CORTES REAIS VS PREVISTOS)")
print("="*65)
print(classification_report(y_test_binario, y_pred_binario, target_names=['Seguro', 'Vulnerável']))

🚀 Treinando o XGBRegressor com novas Features de contexto...

🎯 NOVA SIMULAÇÃO DE SEGMENTAÇÃO CORRIGIDA (CORTES DINÂMICOS)
⚙️ Novas réguas calculadas para a sua população:
   🟢 Até 4.70 pontos -> Baixo Risco
   🟡 De 4.70 até 5.15 pontos -> Médio Risco
   🔴 Acima de 5.15 pontos -> Alto Risco

🔹 Baixo Risco (Verde): 1989 usuários (50.00%)
🔹 Médio Risco (Amarelo): 1392 usuários (34.99%)
🔹 Alto Risco (Vermelho): 597 usuários (15.01%)

📊 MÉTRICAS DE CLASSIFICAÇÃO EQUIVALENTES (CORTES REAIS VS PREVISTOS)
              precision    recall  f1-score   support

      Seguro       0.51      0.51      0.51      1989
  Vulnerável       0.51      0.51      0.51      1989

    accuracy                           0.51      3978
   macro avg       0.51      0.51      0.51      3978
weighted avg       0.51      0.51      0.51      3978

